In [ ]:
!git clone https://github.com/RomanShamailov/codebert_tta_detection.git

%cd codebert_tta_detection

In [ ]:
!pip install -r requirements.txt

In [ ]:
import wandb

wandb.login(key="YOUR_KEY")

In [ ]:
DATASET_CONFIG = "gptsniffer_aigcodeset_test"
PROJECT_NAME = "gptsniffer-tta-detection"

TENT_LRS = [1e-4, 1e-5, 1e-6]
TENT_STEPS = [1, 3, 5, 10]
TENT_MODES = {
    "offline": True,   # reset weights before every batch
    "online": False,   # accumulate adaptation across batches
}

CENTROID_LOGIT_SCALES = [1.0, 5.0, 10.0, 20.0, 50.0]
CENTROID_DISTANCES = ["cosine", "euclidean"]

In [ ]:
import os
from IPython import get_ipython

os.environ["HYDRA_FULL_ERROR"] = "1"
ipython = get_ipython()

def run_command(command):
    print("Running:", command, flush=True)
    result = ipython.system(command)
    if result not in (None, 0):
        raise RuntimeError(f"Command failed: {command}")


commands = []

for mode_name, reset_each_batch in TENT_MODES.items():
    for lr in TENT_LRS:
        for steps in TENT_STEPS:
            lr_name = str(lr).replace("-", "m").replace(".", "p")
            run_name = f"aigcodeset_tent_{mode_name}_lr{lr_name}_steps{steps}"
            commands.append(" ".join([
                "python -u inference.py",
                f"datasets={DATASET_CONFIG}",
                "tta=tent",
                f"tta.lr={lr}",
                f"tta.steps={steps}",
                f"tta.reset_each_batch={str(reset_each_batch).lower()}",
                "writer=wandb",
                f"writer.project_name={PROJECT_NAME}",
                f"writer.run_name={run_name}",
                f"inferencer.save_path={run_name}",
            ]))

for distance in CENTROID_DISTANCES:
    for logit_scale in CENTROID_LOGIT_SCALES:
        scale_name = str(logit_scale).replace(".", "p")
        run_name = f"aigcodeset_centroids_{distance}_scale{scale_name}"
        commands.append(" ".join([
            "python -u inference.py",
            f"datasets={DATASET_CONFIG}",
            "tta=centroids",
            f"tta.distance={distance}",
            f"tta.logit_scale={logit_scale}",
            "writer=wandb",
            f"writer.project_name={PROJECT_NAME}",
            f"writer.run_name={run_name}",
            f"inferencer.save_path={run_name}",
        ]))

print(f"Total runs: {len(commands)}", flush=True)
for index, command in enumerate(commands, start=1):
    print(f"Running {index}/{len(commands)}", flush=True)
    run_command(command)